<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab3_Function_Calling.ipynb)

![Module 3 Lab 3 - Function Calling](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M03/assets/images/M03_Lab3_Function_Calling_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 1 — Function Calling Techniques')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

# 🤖 From Text to Action: Function Calling

**Why this matters.** On its own, a language model is a sealed box. It can only generate text from what it saw during training, so it cannot check today's price, look something up in your database, send an email, run a real calculation, or trigger anything in the outside world. That makes even a brilliant model a dead end for most real software, which has to *act*, not just talk.

**Function calling is the bridge.** It is the one feature that connects a language model to your code, your data, and live systems. You describe the functions ("tools") the model is allowed to use; when a request needs one, the model hands back a structured request to call it, with the arguments already filled in from plain English. You run the function and return the result, and the model finishes its answer using that real data.

This is the foundation of everything people mean by **AI agents** and **AI assistants**. Without function calling an LLM is a clever autocomplete; with it, the model can retrieve, compute, and take action. In this lab we build that mechanism step by step and end with a small assistant that answers real questions by calling real tools.

## ⚠️ 1. The problem: language models cannot compute

A large language model predicts the next **word**, it does not run arithmetic. For small, familiar numbers it often looks right because it has seen similar sums in training. But push it to large or multi-step math and it drifts, and the dangerous part is that it stays confident while being wrong.

In production you cannot ship "probably correct." So the first step is to *see* the failure with our own eyes, which motivates everything after it.

In [ ]:
# ==========================================================
# 1. The model on its own: can it do exact math?
# ==========================================================
# Ask the model a hard multiplication with NO tools, then compare its answer
# to the real value we compute in Python. This exposes the reliability gap.
hard = "What is 48239 * 7854 + 991? Reply with just the number."

r = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": hard}],
)
model_says = r.choices[0].message.content.strip()   # what the model guessed
truth = 48239 * 7854 + 991                          # the real, exact answer

# Show the comparison with pretty_print so it is easy to read
pp({
    "question": hard,
    "model said": model_says,
    "correct answer": truth,
    "did it match?": str(truth) in model_says,      # True only if the model got it exactly
}, title="Trusting the model for math is risky")

## 🔌 How the model learns about your tools

A common misconception is that you "install" or "teach" a function into the model. There is no training and no permanent setup. The connection happens fresh on **every single request**:

1. With each API call you pass a **`tools`** list, one **JSON schema** per function, giving its **name**, a plain-English **description**, and its **parameters** (types and which are required).
2. The model reads those descriptions and, *for that one request*, becomes aware it may call them. It never sees or runs your actual code, only the schema, so the description is all it has to go on.
3. If a tool fits the request, the model does not answer in prose. It replies with a **tool call**: the function name plus the arguments it extracted from the user's words.
4. **Your code** runs the real function and returns the result, and only then does the model write the final answer.

Two consequences worth remembering: because the model is **stateless**, you resend the `tools` list every time; and because the model chooses purely from your **descriptions**, a clear description is what makes it pick the right tool at the right moment, so write it like documentation aimed at the model. With that mental model in place, let's build one.

## 🧮 2. Give the model a calculator tool

A **tool** is just one of your own functions, described to the model with a small **JSON schema**: its name, what it does, and what arguments it takes. The model never runs your code, it only reads the description and, when useful, produces a structured request to call it. You stay fully in control of what the function actually does.

Below we build a **safe** calculator. We deliberately avoid Python's `eval()`, because `eval()` would execute *any* code hidden in the string, a real security hole. Instead we use the `ast` module to parse the text into an expression tree and evaluate only the arithmetic parts, so nothing else can run.

In [ ]:
# ==========================================================
# 2. Define a real calculator tool (safe arithmetic, no eval risk)
# ==========================================================
import ast, operator, json

# Map each kind of math operator in the parsed tree to the Python function
# that actually performs it. Only these operations are allowed to run.
_OPS = {
    ast.Add:  operator.add,       # a + b
    ast.Sub:  operator.sub,       # a - b
    ast.Mult: operator.mul,       # a * b
    ast.Div:  operator.truediv,   # a / b
    ast.Pow:  operator.pow,       # a ** b
    ast.Mod:  operator.mod,       # a % b (remainder)
    ast.USub: operator.neg,       # -a  (negative sign)
}

def _ev(node):
    """Recursively evaluate one node of the parsed expression tree."""
    if isinstance(node, ast.Constant):     # a plain number, e.g. 991
        return node.value
    if isinstance(node, ast.BinOp):        # 'left OP right', e.g. 3 * 4
        return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp):      # a leading minus, e.g. -5
        return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("unsupported expression")   # reject anything that is not arithmetic

def calculate(expression):
    """Evaluate a plain arithmetic expression like '48239 * 7854 + 991'."""
    tree = ast.parse(expression, mode="eval")    # turn the text into a tree
    return _ev(tree.body)                         # evaluate from the top of the tree

# The schema the MODEL reads. It never sees the code above, only this description.
calc_tool = {
    "type": "function",
    "function": {
        "name": "calculate",                       # must match the function name we execute
        "description": "Evaluate an arithmetic expression exactly.",
        "parameters": {                            # describes the arguments the model must supply
            "type": "object",
            "properties": {
                "expression": {"type": "string",
                               "description": "e.g. '48239 * 7854 + 991'"},
            },
            "required": ["expression"],
        },
    },
}

# Quick sanity check that our calculator returns the exact value
pp({"expression": "48239 * 7854 + 991", "result": calculate("48239 * 7854 + 991")},
   title="Our calculator works")

### The call, execute, return loop

Here is the four-step handshake that makes function calling work:

1. You send the question **and** the list of tools.
2. The model replies not with prose but with a **tool call**, the function name plus the arguments it pulled out of your sentence.
3. **You** run the real function with those arguments.
4. You append the result as a `tool` message and ask again, and now the model writes the final answer grounded in the real number.

The model orchestrates; your code executes. Watch the same hard question get an exact answer this time.

In [ ]:
# ==========================================================
# 3. The call -> execute -> return loop (one tool)
# ==========================================================
messages = [{"role": "user", "content": "What is 48239 * 7854 + 991?"}]

# Step 1-2: send the question WITH the tool; the model asks to call it
resp = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL, messages=messages,
    tools=[calc_tool], tool_choice="auto",     # "auto" = the model decides whether to call a tool
)
msg = resp.choices[0].message
tc = msg.tool_calls[0]                          # the specific call the model wants to make
args = json.loads(tc.function.arguments)        # arguments arrive as a JSON string -> dict
pp({"model wants to call": tc.function.name, "arguments": args}, title="The tool call")

# Step 3: WE run the function the model requested
result = calculate(**args)

# Step 4: hand the result back (role "tool") and let the model finish the answer
messages.append(msg)                            # the assistant's tool-call turn
messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
final = client.chat.completions.create(model=DEFAULT_MINI_MODEL, messages=messages)
pretty_print(final.choices[0].message.content, title="Exact answer, delivered via the tool")

## 📡 3. A tool for things the model cannot know

The calculator fixed *reasoning*. Tools are just as important for **knowledge the model simply does not have**: anything real-time, private, or external. A model was trained months ago, so it has no idea about today's prices, your company's database, or your calendar right now.

A tool bridges that gap, it lets the model reach out to a live source and pull a real value into its answer. Here the tool calls **CoinGecko's public API** for a current crypto price. We wrap the request in a small retry so a temporary rate limit (HTTP 429) does not break the lab.

In [ ]:
# ==========================================================
# 4. A real tool: live crypto price (with polite rate-limit retry)
# ==========================================================
import requests, time

def cg_get(url, params, tries=4, timeout=20):
    """GET from CoinGecko, retrying politely if we hit the free-tier rate limit."""
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout)
        if r.status_code == 429:               # 429 = too many requests
            time.sleep(2 ** i)                 # wait 1, 2, 4, 8 seconds, then retry
            continue
        r.raise_for_status()                   # raise on any other error
        return r
    r.raise_for_status()
    return r

def get_crypto_price(coin):
    """Return the live USD price and 24h change for a coin id, e.g. 'bitcoin'."""
    data = cg_get(
        "https://api.coingecko.com/api/v3/simple/price",
        {"ids": coin, "vs_currencies": "usd", "include_24hr_change": "true"},
    ).json()[coin]                             # pull this coin's entry out of the JSON
    return {"coin": coin, "price_usd": data["usd"],
            "change_24h_pct": round(data.get("usd_24h_change", 0), 2)}

# The schema the model reads for this tool
price_tool = {
    "type": "function",
    "function": {
        "name": "get_crypto_price",
        "description": "Get the current USD price and 24h change for a crypto coin.",
        "parameters": {
            "type": "object",
            "properties": {
                "coin": {"type": "string", "description": "coin id, e.g. bitcoin, ethereum, solana"},
            },
            "required": ["coin"],
        },
    },
}

# Call it directly (no model yet) to prove the tool returns live data
pp(get_crypto_price("bitcoin"), title="Live price, straight from the tool")

## 🔗 4. Chaining tools: your first agent

So far the model made a single call. The real leap is to hand it **every** tool and let it **loop**: read the question, call a tool, look at the result, decide whether it needs another tool, and stop only when it can answer. That "call, observe, decide" cycle, run automatically, is exactly what an **AI agent** is.

Our showcase question needs *two* tools in order, first fetch the live Bitcoin price, then compute a percentage of it. Notice we never tell the model which tools to use or in what order; it works that out on its own from the tool descriptions.

In [ ]:
# ==========================================================
# 5. The agent loop: give the model every tool, run until it is done
# ==========================================================
TOOLS = [calc_tool, price_tool]                              # everything the model may use
FUNCTIONS = {"calculate": calculate, "get_crypto_price": get_crypto_price}  # name -> real function

def run_assistant(question, max_steps=6, show=True):
    """Let the model call tools in a loop until it produces a final answer."""
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):                              # a safety cap on the loop
        resp = client.chat.completions.create(
            model=DEFAULT_MINI_MODEL, messages=messages,
            tools=TOOLS, tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg)                                # keep the running conversation
        if not msg.tool_calls:                              # no tool requested -> this is the answer
            return msg.content
        for tc in msg.tool_calls:                           # run EVERY tool the model asked for
            args = json.loads(tc.function.arguments)
            out = FUNCTIONS[tc.function.name](**args)       # look up + execute the real function
            if show:
                print(f"  step: {tc.function.name}({args}) -> {out}")   # trace the loop
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)})  # feed the result back
    return "(stopped: hit the step limit)"

print("Watch the assistant chain two tools:")
answer = run_assistant("What is 15% of the current price of Bitcoin?")
pretty_print(answer, title="Ask anything: a two-tool answer")

> **Pause and think.** For that last question the model had to (1) fetch a **live** price, then (2) do **exact** math on it, then answer, all decided from one English sentence. That call-observe-decide loop is precisely how AI agents operate. Where in your own work is a question really "look something up, then compute, then answer"?

**Your notes** *(double-click to edit)*

- A real question that needs look-up **and** compute: 
- Which tools it would need: 
- What could go wrong if the model chose the wrong arguments: 

## 🔧 5. Hands-on: add your own tool

Now extend the assistant. Write one new function, describe it as a tool, register it in both `FUNCTIONS` and `TOOLS`, then ask a question that should trigger it. Ideas: `convert_currency(amount, from_ccy, to_ccy)`, `days_between(date1, date2)`, or `word_count(text)`. Replace each `-----` with the right value.

In [ ]:
# ==========================================================
# 6. Hands-on: add a tool to the assistant (fill in the -----)
# ==========================================================
def my_function(-----):                 # your parameter name(s)
    """Describe what your tool does."""
    return -----                        # return a value or a small dict

my_tool = {
    "type": "function",
    "function": {
        "name": "-----",                # must match the function name above
        "description": "-----",         # a clear one-line description helps the model choose it
        "parameters": {
            "type": "object",
            "properties": {
                "-----": {"type": "-----", "description": "-----"},
            },
            "required": ["-----"],
        },
    },
}

# Register your tool with the assistant, then ask a question that needs it
FUNCTIONS["-----"] = my_function
TOOLS.append(my_tool)
pretty_print(run_assistant("-----"), title="Your tool in action")

## 🎯 Wrap-up

You built the whole path from text to action:

- **Define** a tool with a JSON schema, the model reads it but never runs it.
- The model **requests** a call with arguments pulled from plain English.
- **You execute** the real function and return the result.
- The model **continues** with a real, grounded answer, and in the agent loop it repeats this across several tools until the question is fully solved.

Combined with JSON mode and Pydantic from Lab 2, you now control the entire pipeline: what goes in (the prompt), how it comes out (JSON), whether it is valid (Pydantic), and what the model can *do* (functions). That call-observe-decide loop is the foundation of the **AI agents** you build in the next module.